# 01 · Prepare the fine-tuning dataset

Downloads `nvidia/Daring-Anteater` from the Hugging Face Hub, takes a stratified subset, formats it into `nvidia/Nemotron-Mini-4B-Instruct`'s chat template, and saves a train/eval split to disk for the next notebook.

Run this on a CPU-only Colab runtime — no GPU needed for this part.

In [ ]:
!pip install -q "datasets>=2.19" "transformers>=4.44" huggingface_hub

In [ ]:
from huggingface_hub import login

# Needs a free HF account + a token from https://huggingface.co/settings/tokens
# (read scope is enough for this notebook).
login()

In [ ]:
from datasets import load_dataset

SUBSET_SIZE = 4000   # keep a T4 fine-tune run under ~1 hour; raise if you have more GPU time
SEED = 42

raw = load_dataset("nvidia/Daring-Anteater", split="train")
print(f"Full dataset: {len(raw)} examples")

raw = raw.shuffle(seed=SEED)
subset = raw.select(range(min(SUBSET_SIZE, len(raw))))
print(f"Working subset: {len(subset)} examples")
print(subset[0])

In [ ]:
from transformers import AutoTokenizer

BASE_MODEL = "nvidia/Nemotron-Mini-4B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)


def to_chat_text(example):
    """Daring-Anteater stores a `conversations` list of {from, value} turns.
    Convert it into the base model's chat template so SFTTrainer can train
    directly on the resulting string.
    """
    role_map = {"human": "user", "gpt": "assistant", "system": "system"}
    turns = example["conversations"]
    messages = [
        {"role": role_map.get(t["from"], "user"), "content": t["value"]}
        for t in turns
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {"text": text}


formatted = subset.map(to_chat_text, remove_columns=subset.column_names)
print(formatted[0]["text"][:500])

In [ ]:
split = formatted.train_test_split(test_size=0.05, seed=SEED)
train_ds, eval_ds = split["train"], split["test"]
print(f"train: {len(train_ds)}  eval: {len(eval_ds)}")

train_ds.save_to_disk("/content/daring_anteater_train")
eval_ds.save_to_disk("/content/daring_anteater_eval")
print("Saved. Continue in 02_lora_finetune.ipynb (switch runtime to a T4 GPU first).")